# MobileNetV2 INT8 com AveragePool2D explícito

Este notebook carrega `best__model.pt`, substitui o pooling somente no modelo de exportação e gera novos artefatos sem alterar os modelos existentes.

Antes de executar, ajuste `DATASET_ROOT` e `PROJECT_DIR`. O candidato só deve ser usado no ESP32 se `BUILTIN_WITHOUT_DEFAULT_DELEGATES` e `BUILTIN_REF` apresentarem acurácias próximas.


In [2]:
import random
from pathlib import Path

import litert_torch
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from ai_edge_litert.interpreter import Interpreter, OpResolverType
from ai_edge_quantizer import quantizer, recipe
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [3]:
PROJECT_DIR = Path.cwd()
PATH = "/home/jovyan/work/UFSC/butterflies_austria/Dataset_train_test_split/butterflies-austria"
DATASET_ROOT = Path(PATH)
CHECKPOINT_PATH = PROJECT_DIR / "best__model.pt"
OUTPUT_DIR = PROJECT_DIR / "tflite_models_avgpool"

NUM_CALIB_SAMPLES = 100
BATCH_SIZE = 64
IMAGE_SIZE = (224, 224)

FP32_PATH = OUTPUT_DIR / "model_fp32_avgpool.tflite"
INT8_PATH = OUTPUT_DIR / "model_int8_avgpool.tflite"
INT8A16_PATH = OUTPUT_DIR / "model_int8a16_avgpool.tflite"

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(DATASET_ROOT)
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(CHECKPOINT_PATH)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {DATASET_ROOT}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Saída: {OUTPUT_DIR}")


Dataset: /home/jovyan/work/UFSC/butterflies_austria/Dataset_train_test_split/butterflies-austria
Checkpoint: /home/jovyan/work/UFSC/butterflies_austria/butterflies_austria_edge_vision/pytorch_mobilevit_xxs/MobileNetV2/butterflies_austria_MobileNetv2/best__model.pt
Saída: /home/jovyan/work/UFSC/butterflies_austria/butterflies_austria_edge_vision/pytorch_mobilevit_xxs/MobileNetV2/butterflies_austria_MobileNetv2/tflite_models_avgpool


In [4]:
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)

input_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.Lambda(lambda image: image.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

train_dataset = datasets.ImageFolder(
    DATASET_ROOT / "train",
    transform=input_transform,
)
test_dataset = datasets.ImageFolder(
    DATASET_ROOT / "test",
    transform=input_transform,
)

if train_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("As classes de treino e teste não estão alinhadas")

class_names = train_dataset.classes
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"Classes: {len(class_names)}")
print(f"Treino: {len(train_dataset)}")
print(f"Teste: {len(test_dataset)}")


Classes: 20
Treino: 5183
Teste: 648


In [5]:
model = timm.create_model(
    "mobilenetv2_100",
    pretrained=False,
    num_classes=len(class_names),
    drop_rate=0.3,
)
state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)
model.load_state_dict(state_dict)
model.eval()

sample_inputs = (test_dataset[0][0].unsqueeze(0),)
with torch.no_grad():
    feature_shape = tuple(model.forward_features(sample_inputs[0]).shape)

if feature_shape[-2:] != (7, 7):
    raise ValueError(f"Feature map esperado: 7x7; recebido: {feature_shape}")

print(f"Feature map: {feature_shape}")


Feature map: (1, 1280, 7, 7)


In [6]:
class MobileNetV2FixedAvgPool(nn.Module):
      def __init__(self, backbone):
          super().__init__()
          self.backbone = backbone

      def forward(self, inputs):
          outputs = self.backbone.forward_features(inputs)
          outputs = F.avg_pool2d(outputs, kernel_size=(7, 7), stride=1)
          outputs = torch.flatten(outputs, 1)
          outputs = F.dropout(
              outputs,
              p=self.backbone.drop_rate,
              training=self.backbone.training,
          )
          return self.backbone.classifier(outputs)


export_model = MobileNetV2FixedAvgPool(model).eval()

with torch.no_grad():
    expected = model(sample_inputs[0])
    actual = export_model(sample_inputs[0])

max_absolute_error = float(torch.max(torch.abs(expected - actual)))
torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
print(f"Erro absoluto máximo: {max_absolute_error:.10f}")


Erro absoluto máximo: 0.0000002384


In [7]:
def operator_names(model_path):
    interpreter = Interpreter(
        model_path=str(model_path),
        experimental_op_resolver_type=(
            OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
        ),
    )
    interpreter.allocate_tensors()
    return [operation["op_name"] for operation in interpreter._get_ops_details()]


def validate_pooling_operator(model_path):
    operators = operator_names(model_path)
    print(f"Operadores em {model_path.name}: {operators}")
    if "SUM" in operators:
        raise RuntimeError(f"{model_path.name} ainda contém SUM")
    if "AVERAGE_POOL_2D" not in operators:
        raise RuntimeError(f"{model_path.name} não contém AVERAGE_POOL_2D")


FP32_PATH.unlink(missing_ok=True)
edge_model = litert_torch.convert(export_model, sample_inputs)
edge_model.export(str(FP32_PATH))
validate_pooling_operator(FP32_PATH)
print(f"FP32: {FP32_PATH.stat().st_size / 1024 / 1024:.2f} MiB")


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/opt/conda/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:02) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:01)

...

(00:09) [ DONE] LiteRT-Torch Convert (+00:09)

(00:00) [START] Write Model to 
/home/jovyan/work/UFSC/butterflies_austria/butterflies_austria_edge_vision/pytorch_mobilevit_xxs/MobileNetV2/butter
flies_austria_MobileNetv2/tflite_models_avgpool/model_fp32_avgpool.tflite

I0000 00:00:1789521154.607041    1526 flatbuffer_export.cc:4342] Estimated count of arithmetic ops: 605.997 M  ops, equivalently 302.998 M  MACs


(00:00) [ DONE] Write Model to 
/home/jovyan/work/UFSC/butterflies_austria/butterflies_austria_edge_vision/pytorch_mobilevit_xxs/MobileNetV2/butter
flies_austria_MobileNetv2/tflite_models_avgpool/model_fp32_avgpool.tflite (+00:00)

Operadores em model_fp32_avgpool.tflite: ['TRANSPOSE', 'PAD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'AVERAGE_POOL_2D', 'FULLY_CONNECTED']
FP32: 8.62 MiB


In [8]:
def create_calibration_data(model_path, dataset, num_samples, seed=SEED):
    interpreter = Interpreter(model_path=str(model_path))
    signatures = interpreter.get_signature_list()
    if len(signatures) != 1:
        raise ValueError(f"Esperada uma assinatura: {list(signatures)}")

    signature_name, signature = next(iter(signatures.items()))
    input_names = signature["inputs"]
    if len(input_names) != 1:
        raise ValueError(f"Esperada uma entrada: {input_names}")

    generator = np.random.default_rng(seed)
    indices = generator.choice(
        len(dataset),
        size=min(num_samples, len(dataset)),
        replace=False,
    )
    input_name = input_names[0]
    samples = []
    for index in indices:
        image = dataset[int(index)][0]
        samples.append({
            input_name: image.unsqueeze(0).numpy().astype(np.float32)
        })
    return {signature_name: samples}


calibration_data = create_calibration_data(
    FP32_PATH,
    train_dataset,
    NUM_CALIB_SAMPLES,
)
print(f"Amostras de calibração: {NUM_CALIB_SAMPLES}")


Amostras de calibração: 100


In [9]:
def quantize_static(fp32_path, output_path, calibration, quantization_recipe):
    qt = quantizer.Quantizer(str(fp32_path))
    qt.load_quantization_recipe(quantization_recipe())
    calibration_result = qt.calibrate(calibration)
    result = qt.quantize(calibration_result)
    result.export_model(str(output_path), overwrite=True)
    print(f"{output_path.name}: {output_path.stat().st_size / 1024 / 1024:.2f} MiB")


quantize_static(
    FP32_PATH,
    INT8A16_PATH,
    calibration_data,
    recipe.static_wi8_ai16,
)
quantize_static(
    FP32_PATH,
    INT8_PATH,
    calibration_data,
    recipe.static_wi8_ai8,
)


/opt/conda/lib/python3.11/site-packages/ai_edge_litert/interpreter.py:480: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
/opt/conda/lib/python3.11/site-packages/ai_edge_quantizer/algorithms/uniform_quantize/uniform_quantize_tensor.py:109: RuntimeWarning: invalid value encountered in cast
  return tensor.astype(qtype, copy=False)
Applying Transformations to tensors:: 100%|██████████| 179/179 [00:00<00:00, 81411.89it/s]


Model name: /home/jovyan/work/UFSC/butterflies_austria/butterflies_austria_edge_vision/pytorch_mobilevit_xxs/MobileNetV2/butterflies_austria_MobileNetv2/tflite_models_avgpool/model_fp32_avgpool.tflite
Original model size: 8.62 MiB
Quantized model size: 2.75 MiB
Quantization Ratio: 0.32 (3.1x smaller)
Total time: 38.80 ms
model_int8a16_avgpool.tflite: 2.75 MiB


Applying Transformations to tensors:: 100%|██████████| 179/179 [00:00<00:00, 89645.42it/s]

Model name: /home/jovyan/work/UFSC/butterflies_austria/butterflies_austria_edge_vision/pytorch_mobilevit_xxs/MobileNetV2/butterflies_austria_MobileNetv2/tflite_models_avgpool/model_fp32_avgpool.tflite
Original model size: 8.62 MiB
Quantized model size: 2.68 MiB
Quantization Ratio: 0.31 (3.2x smaller)
Total time: 34.44 ms
model_int8_avgpool.tflite: 2.68 MiB


In [10]:
validate_pooling_operator(INT8_PATH)
validate_pooling_operator(INT8A16_PATH)


Operadores em model_int8_avgpool.tflite: ['TRANSPOSE', 'PAD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'PAD', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'CONV_2D', 'CONV_2D', 'AVERAGE_POOL_2D', 'FULLY_CONNECTED']
Operadores em model_int8a16_avgpool.tflite: ['TRANSPOSE', 'P

In [11]:
def quantize_input(values, tensor_details):
    dtype = tensor_details["dtype"]
    if not np.issubdtype(dtype, np.integer):
        return values.astype(dtype)

    scale, zero_point = tensor_details["quantization"]
    if scale <= 0:
        raise ValueError("Escala de entrada inválida")

    quantized = np.round(values / scale + zero_point)
    limits = np.iinfo(dtype)
    return np.clip(quantized, limits.min, limits.max).astype(dtype)


def evaluate_tflite(model_path, data_loader, resolver_type, resolver_name):
    interpreter = Interpreter(
        model_path=str(model_path),
        experimental_op_resolver_type=resolver_type,
    )
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    labels = []
    predictions = []

    for images, batch_labels in tqdm(
        data_loader,
        desc=f"{model_path.name} | {resolver_name}",
    ):
        for image, label in zip(images.numpy(), batch_labels.numpy()):
            values = quantize_input(image[None], input_details)
            interpreter.set_tensor(input_details["index"], values)
            interpreter.invoke()
            output = interpreter.get_tensor(output_details["index"])
            labels.append(int(label))
            predictions.append(int(np.argmax(output, axis=1)[0]))

    accuracy = accuracy_score(labels, predictions)
    print(f"{model_path.name} | {resolver_name}: {accuracy:.4f}")
    return accuracy


In [12]:
resolvers = {
    "builtin_without_default_delegates": (
        OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    ),
    "builtin_ref": OpResolverType.BUILTIN_REF,
}
models = {
    "int8": INT8_PATH,
    "int8a16": INT8A16_PATH,
}
results = {}

for model_name, model_path in models.items():
    results[model_name] = {}
    for resolver_name, resolver_type in resolvers.items():
        results[model_name][resolver_name] = evaluate_tflite(
            model_path,
            test_loader,
            resolver_type,
            resolver_name,
        )

print(results)

int8_builtin = results["int8"]["builtin_without_default_delegates"]
int8_reference = results["int8"]["builtin_ref"]
if int8_reference < int8_builtin - 0.01:
    raise RuntimeError(
        "O modelo INT8 ainda diverge entre os resolvers e não deve ir para o ESP32"
    )

print(f"Modelo INT8 aprovado no PC: {INT8_PATH}")


model_int8_avgpool.tflite | builtin_without_default_delegates:   0%|          | 0/11 [00:00<?, ?it/s]

model_int8_avgpool.tflite | builtin_without_default_delegates: 0.9738


model_int8_avgpool.tflite | builtin_ref:   0%|          | 0/11 [00:00<?, ?it/s]

model_int8_avgpool.tflite | builtin_ref: 0.9722


model_int8a16_avgpool.tflite | builtin_without_default_delegates:   0%|          | 0/11 [00:00<?, ?it/s]

model_int8a16_avgpool.tflite | builtin_without_default_delegates: 0.9722


model_int8a16_avgpool.tflite | builtin_ref:   0%|          | 0/11 [00:00<?, ?it/s]

model_int8a16_avgpool.tflite | builtin_ref: 0.9722
{'int8': {'builtin_without_default_delegates': 0.9737654320987654, 'builtin_ref': 0.9722222222222222}, 'int8a16': {'builtin_without_default_delegates': 0.9722222222222222, 'builtin_ref': 0.9722222222222222}}
Modelo INT8 aprovado no PC: /home/jovyan/work/UFSC/butterflies_austria/butterflies_austria_edge_vision/pytorch_mobilevit_xxs/MobileNetV2/butterflies_austria_MobileNetv2/tflite_models_avgpool/model_int8_avgpool.tflite


## Critério de conclusão

O experimento está concluído somente se:

- o modelo FP32 e os modelos quantizados contiverem `AVERAGE_POOL_2D`;
- nenhum deles contiver `SUM`;
- o wrapper PyTorch for numericamente equivalente ao modelo original;
- a diferença de acurácia INT8 entre os dois resolvers for de no máximo um ponto percentual.

Não substitua o modelo do firmware antes de cumprir esses critérios.
